# Competitor Gap Analysis — Open Water
### Open Water — BERTopic + Capped LLM Synthesis

Sam Torres — The SEO Mermaid

---

## Who this notebook is for
You have an OpenAI or Anthropic API key and $10–20 to validate a workflow. You want labeled cluster output and a prioritized gap report — but you want to be deliberate about what you spend.

This notebook uses Sentence Transformers for embeddings (free) and spends your budget only where it matters: LLM synthesis on the top clusters, using a fast, cheap model.

## What you'll get
Everything from Rising Tide, plus:
- Automatic cluster labels (BERTopic keywords + LLM refinement)
- A prioritized gap report (missing / thin / competitive / strong)
- Recommended actions per cluster
- Estimated cost before you run the LLM step

## Budget estimate
At default settings (top 15 clusters, GPT-4o mini or Claude Haiku):
- GPT-4o mini: ~$0.02–0.05
- Claude Haiku: ~$0.01–0.03
Well within a $10–20 project budget.

## What you need
- Two crawl CSVs (your site + competitor)
- One API key (OpenAI or Anthropic)

## Quick start
1. Upload your CSVs
2. Fill in the CONFIGURATION cell
3. Runtime > Run all


## Step 0: Install dependencies

In [ ]:
!pip install sentence-transformers bertopic hdbscan scikit-learn pandas numpy matplotlib seaborn umap-learn -q
!pip install openai anthropic -q
print("Dependencies installed")

## Configuration — edit this cell before running

In [ ]:
# FILE PATHS
YOUR_SITE_CSV  = "your_site.csv"
COMPETITOR_CSV = "competitor_site.csv"

# COLUMN MAPPING
URL_COLUMN   = "Address"
TEXT_COLUMNS = ["Title 1", "Meta Description 1", "H1-1"]

# LABELS
YOUR_SITE_LABEL  = "My Site"
COMPETITOR_LABEL = "Competitor"

# BERTOPIC
MIN_TOPIC_SIZE = 5   # Min pages to form a cluster. Lower = more clusters.
N_TOP_WORDS    = 8   # Keywords per cluster for auto-labeling

# EMBEDDING MODEL (free — saves budget for LLM synthesis)
ST_MODEL_NAME = "all-MiniLM-L6-v2"

# LLM SYNTHESIS — budget-conscious settings
LLM_PROVIDER = "openai"      # "openai" or "anthropic"
OPENAI_API_KEY   = ""
OPENAI_LLM_MODEL = "gpt-4o-mini"   # Cheap and fast — ideal for batch labeling

# Anthropic alternative:
# LLM_PROVIDER      = "anthropic"
# ANTHROPIC_API_KEY = ""
# ANTHROPIC_MODEL   = "claude-haiku-4-5-20251001"

# BUDGET CAP — only run LLM on top N clusters by gap ratio
# This is how you control spend: start with 10-15, expand if results look good
MAX_LLM_CLUSTERS = 15

print("Configuration loaded")

## Step 1: Load and prepare data

In [ ]:
import pandas as pd
import numpy as np

def load_and_prep(filepath, url_col, text_cols, label):
    df = pd.read_csv(filepath, low_memory=False)
    df = df[df[url_col].notna() & (df[url_col].str.strip() != "")].copy()
    if "Content Type" in df.columns:
        df = df[df["Content Type"].str.contains("text/html", na=False)]
    if "Status Code" in df.columns:
        df = df[df["Status Code"] == 200]
    available_cols = [c for c in text_cols if c in df.columns]
    if not available_cols:
        raise ValueError(f"None of {text_cols} found in {filepath}. Check TEXT_COLUMNS config.")
    df["content"] = df[available_cols].fillna("").apply(
        lambda row: " | ".join(v for v in row if str(v).strip()), axis=1
    )
    df = df[df["content"].str.strip() != ""].copy()
    df = df[[url_col, "content"]].rename(columns={url_col: "url"})
    df["source"] = label
    df = df.reset_index(drop=True)
    print(f"  {label}: {len(df)} pages loaded")
    return df

print("Loading data...")
df_yours = load_and_prep(YOUR_SITE_CSV,  URL_COLUMN, TEXT_COLUMNS, YOUR_SITE_LABEL)
df_comp  = load_and_prep(COMPETITOR_CSV, URL_COLUMN, TEXT_COLUMNS, COMPETITOR_LABEL)
df_all   = pd.concat([df_yours, df_comp], ignore_index=True)
print(f"\nTotal pages to embed: {len(df_all)}")

## Step 2: Generate embeddings (free — Sentence Transformers)

In [ ]:
def get_embeddings_st(texts, model_name):
    from sentence_transformers import SentenceTransformer
    print(f"Loading Sentence Transformers model: {model_name}")
    print("(First run downloads ~90MB — cached after that)")
    model = SentenceTransformer(model_name)
    print(f"Embedding {len(texts)} pages...")
    return model.encode(texts, show_progress_bar=True, batch_size=64)
# TIP: Save embeddings to disk to avoid re-running this step
# import numpy as np
# np.save("embeddings.npy", embeddings)
# To reload: embeddings = np.load("embeddings.npy")

texts      = df_all["content"].tolist()
embeddings = get_embeddings_st(texts, ST_MODEL_NAME)
print(f"\nEmbeddings shape: {embeddings.shape}")

## Step 3: Discover topic clusters with BERTopic
BERTopic automatically discovers the number of clusters from your data.
It also generates keyword labels per cluster — which feed directly into the LLM prompt,
making the synthesis step more accurate and cheaper (less content needed per call).

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

print(f"Running BERTopic (min_topic_size={MIN_TOPIC_SIZE})...")

umap_model = UMAP(n_components=5, n_neighbors=15, min_dist=0.0, metric="cosine", random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=MIN_TOPIC_SIZE, min_samples=1,
                         metric="euclidean", cluster_selection_method="eom", prediction_data=True)
vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)

topic_model = BERTopic(umap_model=umap_model, hdbscan_model=hdbscan_model,
                        vectorizer_model=vectorizer_model, top_n_words=N_TOP_WORDS, verbose=True)

topics, _ = topic_model.fit_transform(texts, embeddings)
df_all["cluster"] = topics

topic_info     = topic_model.get_topic_info()
topic_name_map = dict(zip(topic_info["Topic"], topic_info["Name"]))
df_all["topic_name"] = df_all["cluster"].map(topic_name_map).fillna("Outlier")

n_topics   = len(topic_info[topic_info["Topic"] != -1])
n_outliers = len(df_all[df_all["cluster"] == -1])
print(f"\nTopics discovered: {n_topics} | Outliers: {n_outliers}")
print(topic_info[topic_info["Topic"] != -1][["Topic", "Count", "Name"]].to_string(index=False))

## Step 4: Build gap summary

In [ ]:
df_topics = df_all[df_all["cluster"] != -1].copy()

cluster_summary = (
    df_topics.groupby(["cluster", "topic_name", "source"])
    .size().unstack(fill_value=0).reset_index()
)
for col in [YOUR_SITE_LABEL, COMPETITOR_LABEL]:
    if col not in cluster_summary.columns:
        cluster_summary[col] = 0

cluster_summary["total"]     = cluster_summary[YOUR_SITE_LABEL] + cluster_summary[COMPETITOR_LABEL]
cluster_summary["gap_ratio"] = (
    cluster_summary[COMPETITOR_LABEL] / cluster_summary[YOUR_SITE_LABEL].replace(0, 0.1)
).round(2)
cluster_summary = cluster_summary.sort_values("gap_ratio", ascending=False)
print(cluster_summary[["cluster", "topic_name", YOUR_SITE_LABEL, COMPETITOR_LABEL, "gap_ratio"]].to_string(index=False))

## Step 5: Visualize — Breadth & Depth charts

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("Competitor Gap Analysis: Breadth & Depth", fontsize=16, fontweight="bold", y=1.02)

cs = cluster_summary.copy()
cs["short"] = cs["topic_name"].str[:28]
x, w = range(len(cs)), 0.35

ax1 = axes[0]
ax1.bar([i-w/2 for i in x], cs[YOUR_SITE_LABEL],  w, label=YOUR_SITE_LABEL,  color="#028090", alpha=0.85)
ax1.bar([i+w/2 for i in x], cs[COMPETITOR_LABEL], w, label=COMPETITOR_LABEL, color="#F96167", alpha=0.85)
ax1.set_xticks(list(x))
ax1.set_xticklabels(cs["short"], rotation=45, ha="right", fontsize=8)
ax1.set_ylabel("Number of Pages")
ax1.set_title("Depth: Pages per Topic Cluster")
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

ax2 = axes[1]
hm = cs[[YOUR_SITE_LABEL, COMPETITOR_LABEL]].copy()
hm.index = cs["short"]
sns.heatmap(hm.div(hm.max()).fillna(0).T, ax=ax2, cmap="YlOrRd",
            linewidths=0.5, annot=hm.T, fmt="g", annot_kws={"size": 8},
            cbar_kws={"label": "Relative coverage"})
ax2.set_title("Breadth: Coverage Heatmap")
plt.setp(ax2.get_xticklabels(), rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.savefig("competitor_gap_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: competitor_gap_analysis.png")

## Step 6: Topic space map (UMAP)

In [ ]:
from umap import UMAP
import matplotlib.pyplot as plt

print("Running 2D UMAP for visualization (30-60s)...")
reducer_2d = UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
coords = reducer_2d.fit_transform(embeddings)
df_all["umap_x"] = coords[:, 0]
df_all["umap_y"] = coords[:, 1]

fig, ax = plt.subplots(figsize=(13, 8))
site_colors  = {YOUR_SITE_LABEL: "#028090", COMPETITOR_LABEL: "#F96167"}
site_markers = {YOUR_SITE_LABEL: "o", COMPETITOR_LABEL: "^"}

outliers = df_all[df_all["cluster"] == -1] if -1 in df_all["cluster"].values else pd.DataFrame()
if len(outliers) > 0:
    ax.scatter(outliers["umap_x"], outliers["umap_y"],
               c="#CCCCCC", alpha=0.3, s=20, zorder=1, label="Outlier")

for source, group in df_all[df_all["cluster"] != -1].groupby("source"):
    ax.scatter(group["umap_x"], group["umap_y"],
               c=site_colors.get(source, "#888888"), marker=site_markers.get(source, "o"),
               alpha=0.65, s=45, label=source, zorder=2)

for cid, group in df_all[df_all["cluster"] != -1].groupby("cluster"):
    cx, cy = group["umap_x"].mean(), group["umap_y"].mean()
    label = str(topic_name_map.get(cid, f"T{cid}"))[:22]
    ax.annotate(label, (cx, cy), fontsize=8, fontweight="bold", color="#0D1B2A",
                bbox=dict(boxstyle="round,pad=0.25", fc="white", alpha=0.75, ec="#CCCCCC"))

ax.set_title("Topic Space Map: Your Site vs. Competitor", fontsize=14, fontweight="bold")
ax.set_xlabel("UMAP dimension 1")
ax.set_ylabel("UMAP dimension 2")
ax.legend(markerscale=1.5)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig("topic_space_map.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: topic_space_map.png")


## Step 7: Collect cluster samples

In [ ]:
SAMPLE_PER_CLUSTER = 5
cluster_samples = {}

for cluster_id in sorted(df_topics["cluster"].unique()):
    cluster_pages = df_topics[df_topics["cluster"] == cluster_id]
    topic_name = topic_name_map.get(cluster_id, f"Topic {cluster_id}")
    sample = cluster_pages.sample(min(SAMPLE_PER_CLUSTER, len(cluster_pages)), random_state=42)
    your_count = len(cluster_pages[cluster_pages["source"] == YOUR_SITE_LABEL])
    comp_count = len(cluster_pages[cluster_pages["source"] == COMPETITOR_LABEL])
    topic_words = topic_model.get_topic(cluster_id)
    keywords = ", ".join([w for w, _ in topic_words[:6]]) if topic_words else "N/A"
    cluster_samples[cluster_id] = {
        "topic_name": topic_name, "keywords": keywords,
        "your_count": your_count, "comp_count": comp_count,
        "gap_ratio": round(comp_count / max(your_count, 0.1), 2),
        "sample_content": sample["content"].tolist()
    }

print(f"Collected samples for {len(cluster_samples)} clusters")

## Step 8: LLM Synthesis — capped at top clusters
We only call the LLM for the top clusters by gap ratio.
This keeps costs predictable while covering your highest-priority opportunities.

**Estimated cost at default settings (15 clusters, gpt-4o-mini): ~$0.02–0.05**

In [ ]:
def call_llm(prompt, provider, api_key, model):
    if provider == "openai":
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
        r = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2
        )
        return r.choices[0].message.content.strip()
    elif provider == "anthropic":
        import anthropic
        client = anthropic.Anthropic(api_key=api_key)
        r = client.messages.create(
            model=model,
            max_tokens=600,
            messages=[{"role": "user", "content": prompt}]
        )
        return r.content[0].text.strip()
    else:
        raise ValueError(f"Unknown provider: {provider}")

import json, time

def build_gap_prompt(cluster_id, topic_name, keywords, your_count, comp_count, sample_content, your_label, comp_label):
    preview = "\n".join([f"- {c[:180]}" for c in sample_content[:4]])
    return f"""You are an SEO strategist analyzing a BERTopic cluster.

CLUSTER {cluster_id}:
- Label: {topic_name}
- Keywords: {keywords}
- {your_label} pages: {your_count}
- {comp_label} pages: {comp_count}

Sample content:
{preview}

Return ONLY this JSON:
{{
  "cluster_topic": "2-5 word label",
  "description": "One sentence describing this cluster",
  "gap_type": "missing" | "thin" | "competitive" | "strong",
  "gap_type_explanation": "One sentence",
  "priority": "high" | "medium" | "low",
  "recommended_action": "One concrete action"
}}

Gap types: missing={your_label} has 0/few pages vs significant competitor coverage | thin={your_label} has some but competitor has 2x+ | competitive=similar coverage | strong={your_label} equal or greater
Return ONLY the JSON."""


api_key = OPENAI_API_KEY if LLM_PROVIDER == "openai" else globals().get("ANTHROPIC_API_KEY", "")
model   = OPENAI_LLM_MODEL if LLM_PROVIDER == "openai" else globals().get("ANTHROPIC_MODEL", "")

if not api_key:
    print("No API key set — skipping LLM synthesis.")
    print("Add OPENAI_API_KEY or ANTHROPIC_API_KEY in the config cell.")
else:
    # Sort by gap_ratio, take top MAX_LLM_CLUSTERS
    sorted_clusters = sorted(cluster_samples.items(), key=lambda x: x[1]["gap_ratio"], reverse=True)
    to_label = sorted_clusters[:MAX_LLM_CLUSTERS]
    print(f"Running LLM on top {len(to_label)} clusters (of {len(sorted_clusters)} total)")
    print(f"Skipping {len(sorted_clusters) - len(to_label)} lower-priority clusters to save budget\n")

    results = []
    for cluster_id, data in to_label:
        print(f"  Labeling: {data['topic_name'][:40]}...", end=" ")
        try:
            prompt = build_gap_prompt(
                cluster_id, data["topic_name"], data["keywords"],
                data["your_count"], data["comp_count"], data["sample_content"],
                YOUR_SITE_LABEL, COMPETITOR_LABEL
            )
            raw    = call_llm(prompt, LLM_PROVIDER, api_key, model)
            parsed = json.loads(raw)
            parsed.update({
                "cluster_id": cluster_id, "bertopic_name": data["topic_name"],
                "keywords": data["keywords"], "your_pages": data["your_count"],
                "comp_pages": data["comp_count"]
            })
            results.append(parsed)
            print(f"✅ {parsed['cluster_topic']}")
        except Exception as e:
            print(f"Error: {e}")
        time.sleep(0.3)
    print(f"\nSynthesis complete: {len(results)} clusters labeled")

## Step 9: Gap report

In [ ]:
if not api_key or not results:
    print("No LLM results. Export cluster summary from Step 4.")
    cluster_summary.to_csv("competitor_gap_summary.csv", index=False)
else:
    report_df = pd.DataFrame(results)
    p_ord = {"high": 0, "medium": 1, "low": 2}
    g_ord = {"missing": 0, "thin": 1, "competitive": 2, "strong": 3}
    report_df["ps"] = report_df["priority"].map(p_ord)
    report_df["gs"] = report_df["gap_type"].map(g_ord)
    report_df = report_df.sort_values(["ps", "gs"]).drop(columns=["ps", "gs"])

    print("\n" + "="*70)
    print("COMPETITOR GAP REPORT — PRIORITIZED")
    print("="*70 + "\n")

    for _, row in report_df.iterrows():
        emoji = {"high": "🔴", "medium": "🟡", "low": "🟢"}.get(row["priority"], "⚪")
        print(f"{emoji} [{row['priority'].upper()}] {row['cluster_topic']}")
        print(f"   {row['description']}")
        print(f"   Gap: {row['gap_type'].upper()} — {row['gap_type_explanation']}")
        print(f"   Pages: {YOUR_SITE_LABEL}: {row['your_pages']} | {COMPETITOR_LABEL}: {row['comp_pages']}")
        print(f"   Action: {row['recommended_action']}")
        print()

    cols = ["cluster_id", "cluster_topic", "bertopic_name", "keywords",
            "your_pages", "comp_pages", "gap_type", "priority", "recommended_action"]
    report_df[[c for c in cols if c in report_df.columns]].to_csv("competitor_gap_report.csv", index=False)
    df_all[["url", "source", "cluster", "topic_name"]].to_csv("all_pages_clustered.csv", index=False)
    print("Saved: competitor_gap_report.csv, all_pages_clustered.csv")

---
## You're done!

### Output files
| File | What it contains |
|------|-----------------|
| `competitor_gap_analysis.png` | Breadth & depth charts |
| `topic_space_map.png` | UMAP topic landscape |
| `competitor_gap_report.csv` | Prioritized gap report with LLM labels |
| `all_pages_clustered.csv` | Every page with cluster assignment |

### Was the spend worth it?
If the gap report surfaced 3–5 clear content priorities you didn't already know about, yes.
If you want deeper analysis (coverage quality, redundancy flags, full cluster coverage),
move to **Deep Sea**.

---
*Sam Torres — The SEO Mermaid*
